### Environment Helpers

Small post-ingestion environment-population helpers that need to run after
`Canonical_Data` (catalog + schemas exist) and `Document_Data` (PDFs are in
volumes) but are too small or too miscellaneous to warrant their own stage.

**Current helpers**:

- **AI SQL demo on food-safety inspection PDFs** — exercises
  `ai_parse_document`, `ai_classify`, `ai_extract`, and `ai_summarize`
  against a small sample (`PDF_SAMPLE_SIZE` rows) from
  `/Volumes/{CATALOG}/food_safety/reports/` and writes the results to
  `{CATALOG}.food_safety.ai_*_inspections` tables for inspection from the
  SQL editor / a downstream dashboard.

Add new helpers in their own cells below.  Keep each idempotent
(`CREATE OR REPLACE`, `CREATE … IF NOT EXISTS`, `MERGE`) and best-effort
(skip gracefully if the inputs it needs are missing) so this stage can be
re-run safely as part of a `bundle run caspers`, and so it can be added to
targets that don't run `Document_Data` without breaking them.

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

# Size of the AI SQL demo sample.  `ai_parse_document` bills per page and
# `ai_classify`/`ai_extract`/`ai_summarize` bill per row, so keep this small
# unless you are deliberately running a wider sweep.
PDF_SAMPLE_SIZE = 6
PDF_VOLUME_PATH = f"/Volumes/{CATALOG}/food_safety/reports"

print(f"environment_helpers: catalog={CATALOG}")
print(f"environment_helpers: PDF sample size for AI SQL demo = {PDF_SAMPLE_SIZE}")
print(f"environment_helpers: PDF volume path = {PDF_VOLUME_PATH}")

#### AI SQL demo on inspection PDFs

Reads `PDF_SAMPLE_SIZE` PDFs out of the food-safety `reports` volume and
chains four AI SQL functions over them:

| Function | What we get out | Output table |
|---|---|---|
| `ai_parse_document` | Per-PDF structured `VARIANT` + flattened text | `food_safety.ai_parsed_inspections` |
| `ai_classify` | Bucketed overall risk + dominant violation category | `food_safety.ai_classified_inspections` |
| `ai_extract` | Structured fields (date, location, score, grade) | `food_safety.ai_extracted_inspections` |
| `ai_summarize` | Short plain-English summary per inspection | `food_safety.ai_summarized_inspections` |

All cells are idempotent (`CREATE OR REPLACE`) and gated by the volume
check below so this stage stays safe in targets that don't run
`Document_Data`.

In [ ]:
# Guard: skip the entire AI SQL demo if the food-safety PDF volume is
# absent or empty.  This lets Environment_Helpers be added to targets that
# don't run Document_Data without making the task fail.
try:
    pdf_entries = [
        f for f in dbutils.fs.ls(PDF_VOLUME_PATH)
        if f.name.lower().endswith(".pdf")
    ]
    pdf_count = len(pdf_entries)
except Exception as e:
    print(
        f"⏭️  Cannot access {PDF_VOLUME_PATH} "
        f"({type(e).__name__}: {e}) — skipping AI SQL demo."
    )
    dbutils.notebook.exit("ai_sql_demo: skipped (volume missing)")

if pdf_count == 0:
    print(f"⏭️  No PDFs in {PDF_VOLUME_PATH} — skipping AI SQL demo.")
    dbutils.notebook.exit("ai_sql_demo: skipped (volume empty)")

print(f"✅ Found {pdf_count} PDFs in {PDF_VOLUME_PATH}")
print(f"   Will process the first {min(pdf_count, PDF_SAMPLE_SIZE)} via AI SQL.")

In [ ]:
# ai_parse_document → one row per PDF, with:
#   - the full structured VARIANT (kept for power users)
#   - a flat `text` column that concatenates every text/heading/caption
#     element in document order (the input the downstream AI SQL calls use)
#   - a couple of metadata columns for sanity checks
#
# Cost note: ai_parse_document bills per page, so we LIMIT the source to
# PDF_SAMPLE_SIZE rows.  Remove the LIMIT (and reconsider the volume scan)
# for a production parse pass.
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.food_safety.ai_parsed_inspections
COMMENT 'Inspection PDFs parsed with ai_parse_document; one row per PDF.'
AS
WITH parsed AS (
  SELECT
    element_at(split(path, '/'), -1)                       AS pdf_name,
    ai_parse_document(content, map('version', '2.0'))      AS doc
  FROM READ_FILES('{PDF_VOLUME_PATH}/', format => 'binaryFile')
  WHERE lower(path) LIKE '%.pdf'
  ORDER BY path
  LIMIT {PDF_SAMPLE_SIZE}
)
SELECT
  pdf_name,
  doc                                                       AS parsed,
  array_join(
    transform(
      filter(
        try_variant_get(
          doc,
          '$.document.elements',
          'ARRAY<STRUCT<type STRING, content STRING>>'
        ),
        e -> e.type IN ('text', 'title', 'section_header', 'caption')
      ),
      e -> e.content
    ),
    '\\n'
  )                                                         AS text,
  size(
    try_variant_get(doc, '$.document.pages', 'ARRAY<STRUCT<id INT>>')
  )                                                         AS num_pages,
  CAST(doc:metadata:file_metadata:file_size AS BIGINT)      AS file_size_bytes
FROM parsed
""")

print(f"✅ Created {CATALOG}.food_safety.ai_parsed_inspections")
display(
    spark.sql(f"""
        SELECT pdf_name, num_pages, file_size_bytes, left(text, 200) AS text_preview
        FROM {CATALOG}.food_safety.ai_parsed_inspections
        ORDER BY pdf_name
    """)
)

In [ ]:
# ai_classify → assign each PDF to one of a fixed label set.  Two passes:
#   - overall_risk: bucketed severity of the inspection as a whole
#   - dominant_category: which violation category dominates the report
#
# Labels are chosen to match the categories the generator actually emits
# (see data/inspections/generate_inspection_reports.py) so the demo is
# interpretable against the ground-truth violations table.
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.food_safety.ai_classified_inspections
COMMENT 'Per-PDF risk bucket + dominant violation category from ai_classify.'
AS
SELECT
  pdf_name,
  ai_classify(
    text,
    ARRAY(
      'clean_pass',
      'minor_violations_only',
      'major_violations_present',
      'critical_violations_present'
    )
  ) AS overall_risk,
  ai_classify(
    text,
    ARRAY(
      'Temperature Control',
      'Cross-Contamination',
      'Personal Hygiene',
      'Sanitation',
      'Personnel',
      'Facilities',
      'Equipment',
      'Food Labeling',
      'Maintenance',
      'Pest Control',
      'Administrative',
      'Chemical Safety'
    )
  ) AS dominant_violation_category
FROM {CATALOG}.food_safety.ai_parsed_inspections
WHERE text IS NOT NULL
""")

print(f"✅ Created {CATALOG}.food_safety.ai_classified_inspections")
display(
    spark.sql(f"""
        SELECT * FROM {CATALOG}.food_safety.ai_classified_inspections
        ORDER BY pdf_name
    """)
)

In [ ]:
# ai_extract → pull the structured fields the inspection PDFs print on
# their cover page out of the free-text the parser produced.  The result is
# a STRUCT<field STRING, ...> where each label in the array becomes a
# top-level field.
#
# These should track closely with the canonical values in
# {CATALOG}.food_safety.inspections — comparing the two is a quick way to
# eyeball how well the parse-then-extract chain is doing.
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.food_safety.ai_extracted_inspections
COMMENT 'Structured fields lifted from each inspection PDF via ai_extract.'
AS
SELECT
  pdf_name,
  ai_extract(
    text,
    ARRAY(
      'inspection_date',
      'inspector_name',
      'location_name',
      'address',
      'score',
      'grade',
      'total_violations'
    )
  ) AS extracted
FROM {CATALOG}.food_safety.ai_parsed_inspections
WHERE text IS NOT NULL
""")

print(f"✅ Created {CATALOG}.food_safety.ai_extracted_inspections")
display(
    spark.sql(f"""
        SELECT
          pdf_name,
          extracted.inspection_date,
          extracted.inspector_name,
          extracted.location_name,
          extracted.score,
          extracted.grade,
          extracted.total_violations
        FROM {CATALOG}.food_safety.ai_extracted_inspections
        ORDER BY pdf_name
    """)
)

In [ ]:
# ai_summarize → a short plain-English summary per PDF.  The second arg
# is the max-words target (the function treats it as a soft bound).
#
# Useful as a "what's in this report" tooltip in dashboards or as the
# first column a reviewer sees when triaging a stack of inspections.
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.food_safety.ai_summarized_inspections
COMMENT 'Short plain-English summary of each inspection PDF via ai_summarize.'
AS
SELECT
  pdf_name,
  ai_summarize(text, 60) AS summary
FROM {CATALOG}.food_safety.ai_parsed_inspections
WHERE text IS NOT NULL
""")

print(f"✅ Created {CATALOG}.food_safety.ai_summarized_inspections")
display(
    spark.sql(f"""
        SELECT * FROM {CATALOG}.food_safety.ai_summarized_inspections
        ORDER BY pdf_name
    """)
)

print()
print("✅ Environment_Helpers complete — AI SQL demo tables written to "
      f"{CATALOG}.food_safety.ai_*_inspections")